# 1. Setup and Imports libraries

In [11]:
# 1. Import libraries
import os
import requests
import datetime
import time
import json
import pandas as pd

# 2. Setup Session
session = requests.Session()

# Bắt buộc phải có User-Agent cụ thể để Reddit không block (Lỗi 429)
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json'
}
session.headers.update(headers)

print("✅ Session created with headers!")

✅ Session created with headers!


In [12]:
# Define BASE_DIR if not already set
BASE_DIR = globals().get("BASE_DIR", os.getcwd())
BASE_DIR = r"D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\reddit-results"

if not os.path.exists(BASE_DIR):
    os.makedirs(BASE_DIR)
    print(f"📁 Đã setup thư mục mới tại: {BASE_DIR}")
else:
    print(f"✅ Thư mục đã sẵn sàng tại: {BASE_DIR}")

# (Tùy chọn) In ra đường dẫn tuyệt đối để bạn dễ dàng biết chính xác file đang lưu ở đâu trên máy
print("Absolute Output folder:", os.path.abspath(BASE_DIR))

✅ Thư mục đã sẵn sàng tại: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\reddit-results
Absolute Output folder: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\reddit-results


# 2. Helper functions

In [13]:
def convert_utc_to_vn(timestamp):
    """Chuyển đổi timestamp sang múi giờ Việt Nam"""
    if not timestamp:
        return None
    dt_utc = pd.to_datetime(timestamp, unit='s', utc=True)
    dt_vn = dt_utc.tz_convert("Asia/Ho_Chi_Minh").tz_localize(None)
    return dt_vn

def extract_comments(comment_list, parent_id=None):
    """Hàm đệ quy bóc tách toàn bộ comment và reply từ JSON của Reddit"""
    extracted = []
    
    for item in comment_list:
        # Bỏ qua các object loại 'more' (nút "tải thêm bình luận" của Reddit)
        if item.get('kind') == 'more':
            continue
            
        data = item.get('data', {})
        
        # Bỏ qua nếu comment bị xóa hoặc tác giả bị xóa
        if data.get('author') == '[deleted]' or data.get('body') == '[removed]':
            continue
            
        cid = data.get('id')
        
        # Tạo record cho comment hiện tại
        comment_info = {
            'comment_id': cid,
            'parent_id': parent_id,
            'is_reply': 1 if parent_id else 0,
            'author': data.get('author'),
            'text': data.get('body'),
            'upvotes': data.get('ups'),
            'created_at_vn': convert_utc_to_vn(data.get('created_utc'))
        }
        extracted.append(comment_info)
        
        # Kiểm tra xem comment này có reply (con) không
        replies = data.get('replies')
        if replies and isinstance(replies, dict):
            # Nếu có, tiếp tục gọi đệ quy để lấy các comment con
            child_comments = replies.get('data', {}).get('children', [])
            extracted.extend(extract_comments(child_comments, parent_id=cid))
            
    return extracted

# 3. Function to crawl a single Reddit post

In [14]:
def crawl_reddit_post(post_url, session, sleep=1.5):
    """
    Cào dữ liệu của 1 bài viết và comments.
    Reddit hỗ trợ xuất JSON bằng cách thêm .json vào cuối URL.
    """
    # Xử lý URL (đảm bảo đuôi .json)
    if not post_url.endswith('.json'):
        post_url = post_url.rstrip('/') + '.json'
        
    try:
        time.sleep(sleep) # Tránh bị rate limit
        response = session.get(post_url, timeout=15)
        
        if response.status_code != 200:
            print(f"❌ HTTP Lỗi {response.status_code} tại URL: {post_url}")
            return None, None
            
        json_data = response.json()
        
        # JSON Reddit bài viết luôn trả về 1 list gồm 2 phần tử:
        # [0] là thông tin Bài viết, [1] là danh sách Comments
        post_data = json_data[0]['data']['children'][0]['data']
        comments_data = json_data[1]['data']['children']
        
        # 1. Bóc tách Metadata bài viết
        post_info = {
            'post_id': post_data.get('id'),
            'title': post_data.get('title'),
            'author': post_data.get('author'),
            'subreddit': post_data.get('subreddit'),
            'upvotes': post_data.get('ups'),
            'num_comments': post_data.get('num_comments'),
            'created_at_vn': convert_utc_to_vn(post_data.get('created_utc')),
            'url': post_data.get('url') # Link ảnh/bài viết gốc nếu có
        }
        
        # 2. Bóc tách Comments
        all_comments = extract_comments(comments_data)
        
        print(f"✅ Đã cào thành công bài: {post_info['title'][:50]}... | Lấy được {len(all_comments)} comments.")
        return post_info, all_comments
        
    except Exception as e:
        print(f"⚠️ Lỗi cào dữ liệu bài viết: {e}")
        return None, None

# 4. Save to excel file

In [15]:
def export_reddit_to_excel(post_info, comments_list, out_dir):
    post_id = post_info['post_id']
    
    # Tạo folder con cho bài viết này
    post_folder = os.path.join(out_dir, f"post_{post_id}")
    os.makedirs(post_folder, exist_ok=True)
    
    # 1. Lưu Metadata bài viết ra file JSON (để lưu trữ metadata)
    json_path = os.path.join(post_folder, f"{post_id}_post_meta.json")
    # Cần convert datetime thành string trước khi dump JSON
    post_info_json = post_info.copy()
    post_info_json['created_at_vn'] = str(post_info_json['created_at_vn'])
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(post_info_json, f, ensure_ascii=False, indent=4)
    print(f"✅ Saved JSON: {json_path}")
        
    # 2. Lưu Comments ra Excel
    if comments_list:
        df = pd.DataFrame(comments_list)
        
        # Sắp xếp cột cho đẹp
        cols = ['comment_id', 'parent_id', 'is_reply', 'author', 'text', 'upvotes', 'created_at_vn']
        df = df[cols]
        
        xlsx_path = os.path.join(post_folder, f"{post_id}_comments.xlsx")
        
        with pd.ExcelWriter(xlsx_path, engine="xlsxwriter", datetime_format="yyyy-mm-dd hh:mm:ss") as writer:
            df.to_excel(writer, index=False, sheet_name="comments")
            ws = writer.sheets["comments"]
            ws.freeze_panes(1, 0) # Freeze dòng tiêu đề
            
            # Tự động căn chỉnh độ rộng cột
            for i, col in enumerate(df.columns):
                maxlen = min(80, max(12, df[col].astype(str).str.len().max() if not df.empty else 12))
                ws.set_column(i, i, maxlen + 2)
                
        print(f"✅ Saved Excel: {xlsx_path}")
    else:
        print(f"⚠️ Bài viết này không có bình luận nào để lưu.")

# 5. Main function to crawl multiple Reddit posts

In [16]:
# Cung cấp danh sách các bài viết Reddit bạn muốn cào
TARGET_URLS = [
    "https://www.reddit.com/r/TopCharacterDesigns/comments/1slryxn/the_new_goji_for_godzilla_minus_zero/",
    "https://www.reddit.com/r/learnmachinelearning/comments/1sn85ou/day_8_of_machine_learning/",
    "https://www.reddit.com/r/limbuscompany/comments/1sniyzn/danteh_stop_spaming_that_shit_danteh/",
    "https://www.reddit.com/r/SipsTea/comments/1snvzqd/then_went_on_to_end_up_in_prison_anyway/",
    "https://www.reddit.com/r/ExplainTheJoke/comments/1snff04/what_happened_in_1978/",
    "https://www.reddit.com/r/HistoricalCapsule/comments/1snnewz/106yearold_jack_riddle_and_his_wife_rosie_both/",
    "https://www.reddit.com/r/clevercomebacks/comments/1snvtef/im_sure_the_customers_love_him/",
    "https://www.reddit.com/r/whatisit/comments/1snew0p/parked_at_sfo_long_term/",
    "https://www.reddit.com/r/fuckcars/comments/1snw3o4/new_bridge_in_prague_opens_for_trams_buses_bikes/",
    "https://www.reddit.com/r/mildlyinfuriating/comments/1snnc00/my_wifes_letter_to_our_hoa/"
]

print(f"{'='*60}")
print("🚀 BẮT ĐẦU CÀO DỮ LIỆU REDDIT")
print(f"{'='*60}\n")

for idx, url in enumerate(TARGET_URLS, 1):
    print(f"\n📌 Post {idx}/{len(TARGET_URLS)}")
    print(f"URL: {url}")
    
    post_info, comments = crawl_reddit_post(url, session=session)
    
    if post_info:
        print(f"📁 Lưu dữ liệu vào: reddit-results/post_{post_info['post_id']}/")
        # Xuất dữ liệu ra thư mục
        export_reddit_to_excel(post_info, comments, BASE_DIR)
        
    print("-" * 60)
    
print("\n" + "="*60)
print("🎉 HOÀN THÀNH!")
print("="*60)

🚀 BẮT ĐẦU CÀO DỮ LIỆU REDDIT


📌 Post 1/10
URL: https://www.reddit.com/r/TopCharacterDesigns/comments/1slryxn/the_new_goji_for_godzilla_minus_zero/
✅ Đã cào thành công bài: The new Goji for Godzilla Minus Zero... | Lấy được 48 comments.
📁 Lưu dữ liệu vào: reddit-results/post_1slryxn/
✅ Saved JSON: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\reddit-results\post_1slryxn\1slryxn_post_meta.json
✅ Saved Excel: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\reddit-results\post_1slryxn\1slryxn_comments.xlsx
------------------------------------------------------------

📌 Post 2/10
URL: https://www.reddit.com/r/learnmachinelearning/comments/1sn85ou/day_8_of_machine_learning/
✅ Đã cào thành công bài: Day 8 of Machine Learning:... | Lấy được 14 comments.
📁 Lưu dữ liệu vào: reddit-results/post_1sn85ou/
✅ Saved JSON: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\reddit-results\post_1sn85ou\1sn85ou_post_meta.json
✅ Saved Excel: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\reddit-result